# Week 4 — 딥러닝 아키텍처 비교: Transformer · 3D CNN

## 이번 주의 위치 (먼저 읽기)
지금까지 우리는 pix2pix 기반 모델을 **손실 · 입력 · 적대 학습(GAN)** 으로 개선해 왔다.
구조의 뼈대인 2D 합성곱은 바꾼 적이 없다. 이번 주는 우리 접근이 실제로 좋은지 **확인**하기 위해,
이 문제에 자주 거론되는 다른 구조들(Transformer 계열 · 3D CNN · diffusion)을 같은 조건에서 비교한다.
이 구조들은 우리가 채택하는 개선책이 아니라 **비교 기준(baseline)** 이다.

## 학습 목표
1. attention 연산(Q·K·V, softmax 가중 평균)을 작은 행렬로 직접 계산한다.
2. 창(window) attention 기반 `SwinUNetMini` 와 3D 합성곱 기반 `UNet3DMini` 를 구성한다.
3. 세 비교 구조와 **우리 방식(2D UNet + GAN)** 을 같은 데이터·입력·예산에서 비교한다.
4. 파라미터 수·학습 시간·정확도의 trade-off 를 해석한다.

## 표기 규약
- **Q, K, V**: attention의 질의(query)·색인(key)·내용(value) 행렬. 같은 입력에서 선형 변환으로 만든다.
- **d**: 한 head 의 채널 차원. 점곱 점수를 √d 로 나눠 크기를 안정화한다.
- **W-MSA / SW-MSA**: 고정 창 / 절반 이동(shifted) 창 안에서만 계산하는 multi-head self-attention.
- **k**: 이웃 거리. 슬라이스 t 를 t±k 두 장에서 예측한다.
- **|Δφ|**: 복원과 원본의 공극률 차이(단위 %p). 작을수록 정확하다.
- **SSIM**: 구조 유사도(0~1). 클수록 구조가 유사하다.

> 데이터는 W3와 같은 **Bentheimer 사암**, 이웃 거리도 같은 **k=2**다. 비교군(비교 구조) 세 가지
> `UNetMini`(2D 합성곱) · `SwinUNetMini`(창 attention) · `UNet3DMini`(3D 합성곱)의 파라미터는
> **약 12만 개**로 맞춘다. 여기에 §5에서 **우리 방식(2D UNet + GAN)** 을 같은 예산으로 더해 비교한다.

## 0. 환경 준비

helper 모듈 두 개를 불러온다. W4에서 새로 사용하는 것은 창 attention 관련
(`window_partition`, `WindowAttentionMini`, `SwinBlockMini`, `SwinUNetMini`),
3D 합성곱 관련(`UNet3DMini`, `Slice3DDataset`, `evaluate_model_3d`),
그리고 공정 비교 루프(`benchmark_models`)다.

In [ ]:
import sys
from pathlib import Path

# helper 모듈(dr_utils.py, model_utils.py)은 같은 폴더(다운로드 zip)나 ../helpers(저장소)에 있다.
for _cand in [Path('.'), Path('..') / 'helpers']:
    if (_cand / 'dr_utils.py').exists():
        sys.path.insert(0, str(_cand.resolve())); break

import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

# dr_utils: 데이터 I/O · 선형 보간 baseline · 평가 지표(|Δφ|, SSIM) · 색상 팔레트
from dr_utils import (
    load_volume, porosity, predict_linear_k, eval_targets,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)
# model_utils: W2 UNet + W4에서 추가되는 attention/3D 구조와 공정 비교 루프
from model_utils import (
    UNetMini, count_parameters, evaluate_model,
    # --- W4에서 새로 쓰는 것 ---
    window_partition,        # (B,H,W,C) → 창 단위 분할
    WindowAttentionMini,     # 창 내부 self-attention (attention map 반환 가능)
    SwinBlockMini,           # LN → 창 attention → LN → MLP (+잔차 2회)
    SwinUNetMini,            # 창 attention 기반 mini 모델 (~124K 파라미터)
    UNet3DMini,              # 3D 합성곱 mini 모델 (~129K 파라미터)
    Slice3DDataset,          # 3D 모델용 '공정 입력' 데이터셋
    evaluate_model_3d,       # 3D 모델 평가 (지표는 2D와 동일)
    benchmark_models,        # 같은 데이터·정보·시간 예산의 공정 비교 루프
)
setup_plot_style()   # 한글 폰트 등록 + 그림 기본 스타일 설정

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)   # 재현성을 위한 시드 고정
print(f'PyTorch {torch.__version__}, device={DEVICE}')

## 1. 문제 설정과 기준선

문제는 W2·W3와 같다. 이웃 슬라이스 두 장 `vol[t-k]`, `vol[t+k]` 에서 가운데 슬라이스 `vol[t]` 를 예측한다.
지금까지는 이 문제를 **2D 합성곱(UNet)** 한 가지 구조로만 풀었다. 이번 주 질문은 다음과 같다.

> **구조(연결 방식)를 바꾸면 무엇이 얻어지고, 무엇이 비싸지는가?**

- 합성곱: 각 위치가 주변 3×3 만 보고, 가중치는 내용과 무관하게 고정이다.
- attention: 각 위치가 (창 안의) 모든 위치를 보고, 가중치를 내용의 유사도로 그때그때 계산한다.
- 3D 합성곱: 슬라이스 안(x·y)뿐 아니라 슬라이스 사이(z)도 함께 합성곱한다.

**이 셀에서 보는 것**: 데이터를 로드하고, 비교의 기준선인 선형 보간 지표를 계산한다.

In [ ]:
# Bentheimer 256^3 이진 부피를 로드한다. data/ 또는 ../data/ 에서 파일을 찾는다.
DATA = next((p for p in [Path('data'), Path('..') / 'data'] if (p / 'Bentheimer_256.bin').exists()), Path('data'))
vol = load_volume(DATA / 'Bentheimer_256.bin')
K = 2   # 이웃 거리 k (W3와 동일)
print(f'volume {vol.shape} · 공극률 φ={porosity(vol):.3f} · k={K}')

# 선형 보간 baseline — 학습 없는 기준선
m_lin = eval_targets(predict_linear_k(vol, K), vol, K)
print(f'선형 보간 (k={K})   |Δφ|={m_lin["dphi_pp"]:.2f}%p   SSIM={m_lin["ssim"]:.3f}')

## 2. attention을 숫자로 직접 계산

attention 의 정의는 한 줄이다.

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d}}\right)V$$

말로 풀면: **모든 토큰 쌍의 유사도(QKᵀ)를 재고, √d 로 나눠 안정화한 뒤, softmax 로 합=1 가중치를 만들어, 내용(V)의 가중 평균을 낸다.**

토큰 3개, 차원 d=2 의 작은 예로 전 과정을 직접 계산한다. 토큰은 x₁=[1,0], x₂=[0,1], x₃=[1,1] 이고,
계산을 단순하게 하기 위해 Q=K=X(항등 변환), V=[[10,0],[0,10],[5,5]] 로 둔다.

**읽는 법**: x₃=[1,1] 은 x₁·x₂ 둘 다와 비슷하다. 따라서 3행의 softmax 가중치가 고르게 퍼지고(0.25·0.25·0.50),
출력도 두 내용의 중간값 [5,5] 이 된다. 유사도가 가중치를 만들고, 가중치가 내용을 섞는다.

In [ ]:
X = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])   # 토큰 3개, d=2
Q, Kmat = X, X                                      # 간단히 항등 변환으로 둠
V = torch.tensor([[10., 0.], [0., 10.], [5., 5.]])  # 각 토큰의 '내용'

scores = Q @ Kmat.T                    # 1) 점곱 유사도
scaled = scores / (2 ** 0.5)           # 2) ÷ √d  (d=2)
weights = torch.softmax(scaled, dim=1) # 3) 행마다 합=1 가중치
out = weights @ V                      # 4) 내용의 가중 평균

print('QK^T =\n', scores.numpy())
print('÷√2  =\n', np.round(scaled.numpy(), 2))
print('softmax(행 합=1) =\n', np.round(weights.numpy(), 2))
print('출력 = weights @ V =\n', np.round(out.numpy(), 1))
print('행 합 확인:', weights.sum(dim=1).numpy())
print('√d 제거 시 softmax =\n', np.round(torch.softmax(scores, dim=1).numpy(), 2))  # d=2 라 차이 미미. d 가 크면 §2.2 실측처럼 포화

### 2.1 softmax 유도: 부드러운 argmax 와 기울기

softmax 를 두 각도에서 확인한다. 첫째, 온도 T 를 낮추면 argmax(one-hot)로, 높이면 균등분포로
수렴한다. softmax 는 딱딱한 argmax 의 **미분 가능한 부드러운 버전**이다. 둘째, softmax 의 기울기가
$\partial p_i/\partial s_j = p_i(\delta_{ij}-p_j)$ 라는 깔끔한 형태임을 수치 미분으로 확인한다.
이 단순한 기울기가 softmax 를 딥러닝의 표준 출력으로 만든 이유다.

In [ ]:
def softmax(s):
    e = np.exp(s - s.max()); return e / e.sum()

s = np.array([2.0, 1.0, 0.5, -1.0])
print('온도별 softmax(s/T):')
for T in [3.0, 1.0, 0.3, 0.05]:
    print(f'  T={T:<4} → {np.round(softmax(s / T), 3)}')
print('  → T 를 낮출수록 argmax(one-hot)에 가까워진다')

# 기울기 공식 검증: ∂p_i/∂s_j = p_i(δ_ij − p_j) 를 수치 미분과 비교
p = softmax(s); n = len(s)
J_formula = np.array([[p[i] * ((i == j) - p[j]) for j in range(n)] for i in range(n)])
J_numeric = np.zeros((n, n)); eps = 1e-5
for j in range(n):
    sp = s.copy(); sp[j] += eps
    J_numeric[:, j] = (softmax(sp) - p) / eps
print(f'\n기울기 공식 vs 수치미분 최대 오차: {np.abs(J_formula - J_numeric).max():.2e}  (≈0 이면 공식 확인)')

### 2.2 √d 유도: 점곱 분산이 d 에 비례함을 실측

슬라이드에서 유도한 결과를 숫자로 확인한다. 성분이 평균 0·분산 1인 두 벡터의 점곱
$q\cdot k=\sum_{i=1}^{d} q_i k_i$ 는 항 d 개의 합이라, 기댓값은 0이고 **분산은 d** 가 된다
(표준편차 √d). 즉 차원이 커지면 점수가 저절로 커지고, §2.1에서 본 포화(T→0 상황)가 일어나 기울기가 0이 된다. ÷√d 는 이를 되돌리는 장치다.

**이 셀에서 보는 것**: 여러 차원 d 에서 임의의 q·k 를 많이 뽑아 분산을 재면 d 와 거의 같고,
√d 로 나눈 뒤에는 분산이 다시 1 로 돌아온다.

In [ ]:
rng = np.random.RandomState(0)
print(f'{"d":>5}{"Var[q·k]":>12}{"이론값 d":>10}{"Var(÷√d 후)":>14}')
dims, vars_raw = [], []
for d in [4, 16, 64, 256]:
    q = rng.randn(20000, d); k = rng.randn(20000, d)   # 평균0·분산1 성분
    dot = (q * k).sum(axis=1)                            # 점곱 20000개
    v_raw = dot.var(); v_scaled = (dot / np.sqrt(d)).var()
    dims.append(d); vars_raw.append(v_raw)
    print(f'{d:>5}{v_raw:>12.1f}{d:>10}{v_scaled:>14.3f}')

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(dims, vars_raw, 'o-', color=NAVY, label='실측 Var[q·k]')
ax.plot(dims, dims, '--', color=ORANGE, label='이론값 = d')
ax.set_xlabel('차원 d'); ax.set_ylabel('Var[q·k]'); ax.legend(); ax.grid(alpha=0.15)
ax.set_title('점곱 분산은 차원 d 에 비례한다 → 그래서 ÷√d', fontsize=11)
plt.tight_layout(); plt.show()

### 2.3 창 분할: 비용 문제와 Swin 의 해법

attention 의 비용은 토큰 쌍 수, 즉 **N²** 으로 늘어난다. 256×256 슬라이스의 픽셀을 전부 토큰으로 쓰면
65,536² ≈ 43억 쌍이라 계산이 불가능하다. Swin 은 두 단계로 줄인다.

1. **패치 임베딩**: 4×4 픽셀을 토큰 하나로 요약 → 64×64 = 4,096 토큰.
2. **창 분할**: 토큰 격자를 8×8 창 64개로 나누고, attention 은 **창 안에서만** 계산 → 64 × 64² = 262,144 쌍.

창 밖과의 연결은 다음 블록에서 창을 절반 **이동(shift)** 시켜 만든다. 고정 창과 이동 창을 번갈아 쌓으면
정보가 창 경계를 넘어 전달된다.

**이 셀에서 보는 것**: 실제 슬라이스의 토큰 격자에 8×8 창 경계를 그리고, `window_partition` 의 shape 변화를 확인한다.

In [ ]:
# 비용 계산
n_pix = 256 * 256
n_tok = 64 * 64
print(f'픽셀 전체 attention : {n_pix:,}토큰 → {n_pix**2:,} 쌍')
print(f'패치 임베딩 후      : {n_tok:,}토큰 → {n_tok**2:,} 쌍')
print(f'8×8 창 분할 후      : 64창 × 64² = {64 * 64**2:,} 쌍 (÷{n_tok**2 // (64*64**2)})')

# 실제 슬라이스의 토큰 격자(4×4 평균으로 다운스케일) + 창 경계
sl = vol[128].astype(np.float32)
tokens = sl.reshape(64, 4, 64, 4).mean(axis=(1, 3))
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.imshow(tokens, cmap='gray_r', interpolation='nearest')
for g in range(0, 65, 8):
    ax.axhline(g - 0.5, color=ORANGE, lw=1.2)
    ax.axvline(g - 0.5, color=ORANGE, lw=1.2)
ax.set_title('64×64 토큰 격자와 8×8 창 경계 (z=128)', fontsize=11)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# window_partition 의 shape 변화
x = torch.randn(1, 64, 64, 32)              # (B, H, W, C) 토큰 배열
win = window_partition(x, 8)
print(f'window_partition: {tuple(x.shape)} → {tuple(win.shape)}  (창 64개 × 8×8 × C)')

## 3. SwinUNetMini — 창 attention 기반 모델

`SwinUNetMini` 는 UNet 의 전체 흐름(인코더·디코더·skip)을 유지하되, 합성곱 블록 자리에
**Swin 블록 쌍(고정 창 + 이동 창)** 을 넣은 구조다.

- 패치 임베딩(4×4 conv, stride 4) → 64×64 토큰
- Swin 블록×2 → 다운샘플 → Swin 블록×2 → 업 + skip 결합 → Swin 블록×2 → 업×2
  → **최상단(full-res) skip 결합** → 출력
- 입력·출력 형식은 `UNetMini` 와 완전히 같다: (B, 2, H, W) → (B, 1, H, W)

최상단 skip 은 이 미니 모델에서 특히 중요하다. 이것이 없으면 픽셀 세부를 토큰(1/4 해상도)에서
다시 만들어야 해서, 짧은 학습에서는 "전부 solid(다수 클래스)"라는 자명해에 갇힌다. 실제로 skip
없는 버전은 90초 학습 후 |Δφ| ≈ 공극률(26.7%p), 즉 전부 solid 를 출력했다. 구조의 작은 설계
차이가 학습 성패를 좌우한 사례다.

**이 셀에서 보는 것**: 모델을 만들고 파라미터 수와 forward shape 를 확인한다. 창 attention 의
attention map 도 직접 꺼내 본다(`return_attn=True`). 행 합이 1인 가중치 행렬이 나온다.

In [ ]:
swin = SwinUNetMini(in_ch=2, base=32, num_heads=4, window_size=8)
unet2d = UNetMini(in_ch=2, base=16)
print(f'SwinUNetMini : {count_parameters(swin):,} 파라미터')
print(f'UNetMini(W2) : {count_parameters(unet2d):,} 파라미터  ← 동급으로 맞춤')

# forward shape 확인 (H·W 는 32의 배수)
x = torch.randn(1, 2, 64, 64)
print(f'forward: {tuple(x.shape)} → {tuple(swin(x).shape)}')

# 창 attention 의 attention map 직접 보기
wa = WindowAttentionMini(dim=32, window_size=8, num_heads=4)
tok = torch.randn(1, 64, 32)                 # 창 하나(8×8=64토큰)
out, attn = wa(tok, return_attn=True)
print(f'attention map: {tuple(attn.shape)}  (head 4개 × 64×64 가중치)')
print(f'행 합 = {attn[0, 0].sum(dim=-1)[:4].detach().numpy().round(3)} … (모두 1)')

## 4. UNet3DMini — 3D 합성곱과 '공정 입력'

3D 합성곱은 3×3×3 커널로 슬라이스 안(x·y)과 사이(z)를 함께 본다. 공극은 z 방향으로 연속된
3차원 구조이므로, 부피 문맥을 직접 보는 것은 원리적으로 유리하다.

여기서 **실험 설계의 핵심**이 나온다. 3D 모델에는 부피를 입력하는데, 이때 **2D 모델과 같은 정보**
(이웃 t±k 두 장)만 넣어야 비교가 성립한다.

- 입력: 깊이 2k+1 부피. 채널 0 = 맨 앞·맨 뒤에만 이웃 슬라이스가 채워지고 나머지는 0.
  채널 1 = 채움 위치 마스크.
- 만약 t±1 슬라이스까지 입력에 넣으면 예측 대상 바로 옆의 정답 정보가 새어 들어간다.
  대규모 실험에서 이런 설계 오류 하나가 SSIM 을 0.823 → 0.953 으로 부풀린 사례가 있다.
  **높은 숫자가 좋은 모델을 뜻하지 않는다. 먼저 입력에 무엇이 들어갔는지 확인해야 한다.**

**이 셀에서 보는 것**: `Slice3DDataset` 이 만드는 입력을 눈으로 확인하고(채워진 면 2장 + 빈 면 3장),
모델 파라미터 수와 forward shape 를 확인한다.

In [ ]:
unet3d = UNet3DMini(in_ch=2, base=10)
print(f'UNet3DMini : {count_parameters(unet3d):,} 파라미터  (역시 ~12만)')

# '공정 입력' 확인 — 깊이 5 부피에서 채워진 면은 t±2 두 장뿐
ds3 = Slice3DDataset(vol, k=K, patch_size=64, n_patches_per_triplet=1, augment=False)
x3, y3 = ds3[128]
print(f'입력 {tuple(x3.shape)} = (채널 2, 깊이 {2*K+1}, 64, 64) · target {tuple(y3.shape)}')
print('채널0 각 면의 채움 비율:', x3[0].mean(dim=(1, 2)).numpy().round(3), ' ← 가운데 3장은 0')
print('채널1 마스크        :', x3[1].amax(dim=(1, 2)).numpy(), ' ← 이웃 위치만 1')

fig, axes = plt.subplots(1, 5, figsize=(12, 2.7))
for d in range(5):
    axes[d].imshow(x3[0, d], cmap='gray_r', vmin=0, vmax=1, interpolation='nearest')
    filled = x3[0, d].max() > 0
    axes[d].set_title(f'깊이 {d} · ' + ('이웃 슬라이스' if filled else '빈 면(0)'),
                      fontsize=10, color=NAVY if filled else GRAY)
    axes[d].set_xticks([]); axes[d].set_yticks([])
plt.suptitle('UNet3DMini 입력 채널 0 — 이웃 t±2 두 장만 채워진 부피', fontsize=12, y=1.04)
plt.tight_layout(); plt.show()

# forward: (B, 2, D, H, W) → 가운데 슬라이스 (B, 1, H, W)
xb = torch.randn(1, 2, 2*K+1, 64, 64)
print(f'forward: {tuple(xb.shape)} → {tuple(unet3d(xb).shape)}')

## 5. 공정 비교 — 같은 데이터 · 같은 정보 · 같은 시간 예산

공정 비교의 세 원칙:

1. **같은 데이터**: 같은 volume, 같은 (t−k, t, t+k) triplet.
2. **같은 입력 정보**: 세 모델 모두 이웃 t±k 두 장뿐.
3. **같은 시간 예산**: 모델마다 wall-clock `budget_s` 초. 빠른 모델은 그만큼 epoch 을 더 돈다.

파라미터 수를 맞췄더라도 시간까지 같아야 공정하다. 구조마다 한 스텝의 비용이 다르기 때문이다
(3D 는 깊이 방향 연산이 더해져 한 스텝이 느리다). 시간 예산 비교는 "**같은 컴퓨터로 같은 시간을 쓰면
어느 구조가 유리한가**"라는 실전 질문에 답한다.

**이 셀에서 보는 것**: `benchmark_models` 가 세 비교 구조를 순서대로 학습·평가한다. CPU 에서 약 5분
(학습 90초×3 + 평가) 걸린다. epoch 수가 모델마다 다른 것이 정상이다(같은 시간, 다른 속도).

> **수치는 실행마다 달라진다.** 시간 예산 학습이라 실행 환경(CPU/GPU 속도)에 따라 epoch 수와 지표가
> 매번 다르다. 슬라이드의 수치(예: 우리 방식(GAN) |Δφ| 0.37%p)는 한 실행의 예시다. 절대값이 아니라
> **순위와 경향**(우리 방식 |Δφ| 최저 · 2D UNet SSIM 최고)이 재현되면 정상이다.

In [ ]:
results = benchmark_models(vol, k=K, budget_s=90, device=DEVICE, seed=0)

### 5.1 우리 방식(2D UNet + GAN)을 같은 예산으로 추가

세 비교 구조는 순수한 아키텍처 비교다. 여기에 **우리 방식**, 즉 2D UNet 생성자에 판별자를 붙인
pix2pix GAN(W3)을 같은 90초 예산으로 학습해 함께 놓는다. model_utils 의 `train_gan`(W3)은 epoch 기준 학습이라, 시간 예산 비교에 맞춘 90초 버전 `train_gan_90s` 를 아래에서 정의해 쓴다. 이것이 "우리 모델은 왜 비교에 없나"라는
질문의 실습 답이다. GAN 은 픽셀 SSIM 을 조금 내주는 대신 **물성(공극률) 보존**에서 앞서는 경향을 보인다.

In [ ]:
import time as _time
from model_utils import (PatchDiscriminatorMini, ssim_loss, d_hinge_loss, g_hinge_loss,
                         SliceDataset, UNetMini, evaluate_model, count_parameters)
from torch.utils.data import DataLoader

def train_gan_90s(vol, k, budget_s=90, warmup_s=18, seed=0, device='cpu'):
    torch.manual_seed(seed); np.random.seed(seed)
    G = UNetMini(in_ch=2, base=16).to(device)                 # 우리 생성자 (2D UNet)
    D = PatchDiscriminatorMini(cond_ch=2, base=16).to(device)  # 판별자
    ds = SliceDataset(vol, k=k, patch_size=64, n_patches_per_triplet=2, augment=True)
    loader = DataLoader(ds, batch_size=8, shuffle=True)
    oG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    oD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    t0 = _time.time(); ep = 0; hist = []; stop = False
    import torch.nn.functional as F
    while not stop:
        run, n = 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            lam = 0.0 if (_time.time() - t0) < warmup_s else 0.1   # warmup 후 적대 손실
            yf = G(x)
            if lam > 0:
                oD.zero_grad(); d_hinge_loss(D, x, y, yf).backward(); oD.step()
            oG.zero_grad(); yf = G(x)
            g = F.l1_loss(yf, y) + 0.3 * ssim_loss(yf, y)
            if lam > 0:
                g = g + lam * g_hinge_loss(D, x, yf)
            g.backward(); oG.step()
            run += F.l1_loss(yf, y).item() * x.size(0); n += x.size(0)
            if _time.time() - t0 >= budget_s: stop = True; break
        ep += 1; hist.append((_time.time() - t0, run / max(1, n)))
    ev = evaluate_model(G, vol, k=k, device=device)
    return {'params': count_parameters(G), 'epochs': ep,
            'dphi_pp': ev['dphi_pp'], 'ssim': ev['ssim'],
            'history': hist, 'recon': ev['recon']}

print('우리 방식 (2D UNet + GAN) 학습 중 (약 90초)...')
results['우리 방식(GAN)'] = train_gan_90s(vol, K, budget_s=90, device=DEVICE)
print(f"  → |Δφ|={results['우리 방식(GAN)']['dphi_pp']:.2f}%p · "
      f"SSIM={results['우리 방식(GAN)']['ssim']:.3f}")

### 5.2 학습 곡선: 같은 시간축에서 읽기

**읽는 법**: x축은 epoch 이 아니라 **경과 시간(초)** 이다. 같은 시간 예산에서 각 구조가 손실을
얼마나 빨리 줄이는지가 그대로 보인다. 곡선 점 하나가 1 epoch 이므로, 점 간격이 좁을수록
한 epoch 이 빠른 모델이다.

In [ ]:
colors = {'UNet2D (W2)': NAVY, 'SwinUNet': ORANGE, 'UNet3D': GREEN, '우리 방식(GAN)': '#b26018'}
fig, ax = plt.subplots(figsize=(9.5, 4))
for name, r in results.items():
    t = [h[0] for h in r['history']]; l = [h[1] for h in r['history']]
    ax.plot(t, l, marker='o', ms=3.5, lw=1.6, color=colors[name],
            label=f"{name} · {r['epochs']}ep")
ax.set_xlabel('경과 시간 (s)'); ax.set_ylabel('L1 손실 (epoch 평균)')
ax.legend(); ax.grid(alpha=0.15)
ax.set_title('같은 90초 예산에서의 학습 곡선', fontsize=12)
plt.tight_layout(); plt.show()

## 6. 복원 결과 비교

**읽는 법**: 슬라이스 z=128 의 같은 위치를 원본·선형·세 비교 구조·우리 방식(GAN)으로 본다.
전체 슬라이스에서는 차이가 잘 안 보이므로 64×64 확대(아래)를 함께 본다.
선형 보간은 이웃 평균의 다수결이라 얇은 목(throat)이 끊기거나 뭉개진다.

In [ ]:
zc = 128
lin_rec = predict_linear_k(vol, K)
orig = (vol[zc] > 0).astype(float)
panels = [('원본', orig), (f'선형 (|Δφ|={m_lin["dphi_pp"]:.2f}%p)', lin_rec[zc].astype(float))]
panels += [(f"{n} (|Δφ|={r['dphi_pp']:.2f}%p)", r['recon'][zc]) for n, r in results.items()]

y0, x0 = 96, 96   # 확대 위치
fig, axes = plt.subplots(2, len(panels), figsize=(2.3 * len(panels), 5.6))
for i, (title, img) in enumerate(panels):
    axes[0, i].imshow(img, cmap='gray_r', interpolation='nearest')
    axes[0, i].add_patch(plt.Rectangle((x0, y0), 64, 64, fill=False, color=ORANGE, lw=1.6))
    axes[0, i].set_title(title, fontsize=10)
    axes[1, i].imshow(img[y0:y0+64, x0:x0+64], cmap='gray_r', interpolation='nearest')
    axes[1, i].set_xlabel('64×64 확대', fontsize=9)
    for ax in (axes[0, i], axes[1, i]):
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### 6.1 trade-off 정리

아래 표는 §5의 중간 표에 선형 보간(학습 없는 기준선)과 우리 방식(GAN)을 더한 최종 요약이다.

**읽는 법**: 왼쪽은 파라미터 수 대비 SSIM, 오른쪽은 학습 속도(같은 예산에서 소화한 epoch 수)와
|Δφ| 다. "어느 구조가 이겼는가"보다 **무엇을 내주고 무엇을 얻는가**를 읽는 것이 목적이다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, r in results.items():
    axes[0].scatter(r['params'], r['ssim'], s=90, color=colors[name], zorder=3)
    axes[0].annotate(name, (r['params'], r['ssim']), textcoords='offset points',
                     xytext=(8, 5), fontsize=10)
    axes[1].scatter(r['epochs'], r['dphi_pp'], s=90, color=colors[name], zorder=3)
    axes[1].annotate(name, (r['epochs'], r['dphi_pp']), textcoords='offset points',
                     xytext=(8, 5), fontsize=10)
axes[0].set_xlabel('파라미터 수'); axes[0].set_ylabel('SSIM (높을수록 좋음)')
axes[1].set_xlabel('90초 동안 소화한 epoch 수'); axes[1].set_ylabel('|Δφ| (%p · 낮을수록 좋음)')
for ax in axes: ax.grid(alpha=0.15)
plt.tight_layout(); plt.show()

print(f'{"모델":<14}{"params":>9}{"epochs":>7}{"|Δφ|%p":>9}{"SSIM":>7}')
print(f'{"선형 보간":<14}{"-":>9}{"-":>7}{m_lin["dphi_pp"]:>9.2f}{m_lin["ssim"]:>7.3f}')
for name, r in results.items():
    print(f'{name:<14}{r["params"]:>9,}{r["epochs"]:>7}{r["dphi_pp"]:>9.2f}{r["ssim"]:>7.3f}')

## 7. 심화 — 창 크기 실험

창이 작을수록 attention 쌍 수가 줄어 한 스텝이 빨라지지만, 한 번에 보는 범위도 좁아진다.
`window_size=4` 로 같은 예산 학습을 반복해 본다.

**읽는 법**: 창 크기는 "보는 범위 ↔ 계산 비용"의 손잡이다. 이 데이터·이 예산에서 어느 쪽이
유리한지는 실험으로 확인한다. (같은 방법으로 `base`, `num_heads`, `budget_s` 도 바꿔 볼 수 있다.)

In [ ]:
res_w4 = benchmark_models(
    vol, k=K, budget_s=90, device=DEVICE, seed=0,
    models={'SwinUNet w=4': SwinUNetMini(in_ch=2, base=32, num_heads=4, window_size=4)})

r8, r4 = results['SwinUNet'], res_w4['SwinUNet w=4']
print(f'\n{"":<16}{"창 8×8":>10}{"창 4×4":>10}')
print(f'{"파라미터":<16}{r8["params"]:>10,}{r4["params"]:>10,}')
print(f'{"epoch (90초)":<16}{r8["epochs"]:>10}{r4["epochs"]:>10}')
print(f'{"|Δφ| (%p)":<16}{r8["dphi_pp"]:>10.2f}{r4["dphi_pp"]:>10.2f}')
print(f'{"SSIM":<16}{r8["ssim"]:>10.3f}{r4["ssim"]:>10.3f}')

## 8. 정리 및 다음 주

| 구조 | 연결 방식 | 강점 | 비용 |
|---|---|---|---|
| 2D UNet (W2) | 지역 합성곱, 고정 가중치 | 적은 데이터·시간에서 안정 | 전역 관계는 층을 쌓아야 |
| SwinUNet | 창 attention, 내용 기반 가중치 | 내용에 따라 연결을 바꿈, 파라미터 효율 | 데이터·시간이 더 필요 |
| 3D UNet | 부피 합성곱 | z 연속성을 직접 봄 | 한 스텝이 느림 (같은 시간에 학습량 적음) |

- 같은 크기·같은 시간에서도 **구조에 따라 결과가 달라진다.** 이 문제·이 규모에서는 2D 합성곱의
  귀납 편향(지역 연속성)이 잘 맞는다. 대규모 실험(사암 1000³, 시간 동등)에서도 순서는 같았다.
- **공정 비교의 세 원칙**(같은 데이터·같은 입력 정보·같은 시간 예산)과 **정보 누출 점검**은
  아키텍처 비교의 전제 조건이다.
- 탐구 과제는 핸드아웃 §5 를 참조한다.

**다음 주(W5)**: 구조를 바꾸는 대신, 지금의 2D UNet 을 **쓰는 방법**을 바꾼다. 세 직교축의 예측을
융합(tri-axis)하고, 여러 k 를 함께 학습하고, 증강·후처리로 한계를 보완한다.